# Chavez1996 — DOS

Aplica a aproximação óptica de Chavez (1996), com subtração de haze por objeto escuro (DOS). A saída é uma reflectância corrigida aproximada, adimensional.

## Precisa de

**Entradas para ajustar a correção na cena:**

- **`toa_reflectance` — obrigatório:** imagem óptica em **reflectância TOA, adimensional, em [0,1]**, já ajustada pelo ângulo solar. Não usar DN, radiância ou reflectância superficial já corrigida.
- **`sun_elevation` — obrigatório:** elevação solar da cena, em **graus, no intervalo (0,90]**; obtida do MTL. Não é o ângulo zenital.
- **`percentile` — opcional:** percentil do alvo escuro, na escala **0 a 100**; padrão **0,01**, que significa percentil 0,01 e não 1%.
- **`assumed_dark_surface` — opcional:** reflectância assumida do alvo escuro, **adimensional, em [0,1]**; padrão **0,01**.

A banda óptica demonstrativa é B4. O ajuste deve usar a cena completa válida. Para reutilizá-lo em blocos, fornecer o objeto `parameters` devolvido pelo próprio método e omitir os parâmetros de ajuste, conforme a alternativa abaixo.

**Entrada opcional de qualidade:**

- **`mask`:** imagem **booleana**, sem unidade: `True` exclui o pixel; `False` permite avaliar, sujeito às validações do método. Neste exemplo, sua origem é descrita na seção de leitura.

Todas as imagens usadas juntas devem ter a mesma quantidade de linhas/colunas, posição dos pixels, resolução e sistema de coordenadas.

## Arquivos para executar este notebook

O exemplo usa o leitor de cena **Landsat 8/9 Collection 2 Level 1**. Coloque na pasta da cena:

- **B4:** arquivo original do vermelho, em **DN**; o leitor converte para reflectância TOA, adimensional.
- **B5:** arquivo original do NIR, em **DN**; o leitor converte para reflectância TOA, adimensional.
- **B10:** arquivo térmico original, em **DN**; o leitor converte para BT10 em **K**.
- **B11:** arquivo térmico original, em **DN**; o leitor converte para BT11 em **K**.
- **MTL.txt:** metadados e coeficientes de calibração da própria cena; não é uma imagem.
- **QA_PIXEL:** códigos inteiros de qualidade, sem unidade física; o leitor interpreta os bits para montar a máscara.
- **QA_RADSAT:** códigos inteiros de saturação e oclusão de terreno, sem unidade física; complementam a máscara.

Essa lista corresponde ao leitor usado no notebook; uma chamada direta à função exige somente as entradas indicadas em “Precisa de”. B2 e SZA são opcionais para o leitor. Preserve os nomes originais e a grade dos arquivos. Não passar os arquivos QA brutos como máscara booleana.

## Como as imagens são preparadas

**TOA significa topo da atmosfera.** A grandeza também precisa ser especificada:

- **Radiância TOA:** energia medida pelo sensor, em **W m⁻² sr⁻¹ µm⁻¹**.
- **Reflectância TOA:** **adimensional**, com ajuste do ângulo solar.
- **Temperatura de brilho TOA:** temperatura equivalente da radiância térmica, em **K**.

O leitor usa o **MTL da própria cena** para calibrar os arquivos originais. Não é necessário preparar manualmente as imagens TOA para executar este notebook.

A configuração `optical_quantity="reflectance"` devolve as bandas ópticas em **reflectância TOA, adimensional**; as bandas térmicas são devolvidas como temperaturas de brilho em **K**.

Trocar para `optical_quantity="radiance"` muda a calibração óptica; isso exige verificar a compatibilidade com a parametrização usada no método. **Reflectância TOA ainda contém efeitos atmosféricos e é diferente de reflectância de superfície.**

Edite os caminhos na célula de configuração e execute de cima para baixo. As saídas ficam em `resultados/`; os arquivos originais não são modificados.


## Função geral — parâmetros aplicáveis ao método

Argumentos nomeados e relacionados ficam próximos. A tabela abaixo segue **exatamente a ordem desta chamada**. Obrigatório: necessário ao cálculo; opcional: pode usar o padrão; condicional: depende da rota escolhida. Valores de configuração mostrados podem diferir do padrão da API.

```python
resultado = optical_correction(
    # Método
    method='chavez-dos',

    # Reflectância TOA
    toa_reflectance=opticas[banda],

    # Ajuste na cena completa
    sun_elevation=float(cena.metadata['SUN_ELEVATION']),
    percentile=0.01,
    assumed_dark_surface=0.01,

    # Exclusão de pixels
    mask=mascara,

    # Saída
    return_diagnostics=True,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `"chavez-cost"` | Seletor | Usar "chavez-dos" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `toa_reflectance` | Obrigatório | `Sem padrão` | Reflectância TOA, [0,1] | Imagem já corrigida pelo ângulo solar. Não fornecer DN, radiância ou reflectância de superfície existente. |
| `sun_elevation` | Obrigatório no ajuste; omitir na reutilização | `None` | Graus, (0,90] | Um único ângulo de elevação solar para a cena, obtido do MTL. Não é ângulo zenital. |
| `percentile` | Opcional no ajuste; omitir na reutilização | `None → 0.01` | Percentil, [0,100] | 0.01 significa percentil 0,01, não 1%. Seleção operacional do objeto escuro; ajustar na cena completa válida. |
| `assumed_dark_surface` | Opcional no ajuste; omitir na reutilização | `None → 0.01` | Reflectância, [0,1] | Reflectância assumida do alvo escuro; deve ser compatível com o percentil observado. Não há recorte silencioso de haze negativo. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |
| `return_diagnostics` | Opcional | `False` | Booleano | False retorna somente a imagem; True retorna dicionário com: reflectance, parameters, method, units, invalid_mask. A máscara final pode incluir rejeições adicionais à máscara de entrada. |

## Alternativa — reutilizar o ajuste em blocos

Executar primeiro a chamada principal com return_diagnostics=True sobre a cena completa. bloco e mascara_bloco têm a mesma grade entre si. Omitir os argumentos de ajuste; o objeto parameters já contém o ângulo e o haze.

```python
resultado = optical_correction(
    # Método
    method='chavez-dos',

    # Reflectância TOA
    toa_reflectance=bloco,

    # Ajuste previamente calculado
    parameters=resultado['parameters'],

    # Exclusão de pixels
    mask=mascara_bloco,

    # Saída
    return_diagnostics=False,
)
```

### Detalhamento dos parâmetros

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `"chavez-cost"` | Seletor | Usar "chavez-dos" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `toa_reflectance` | Obrigatório | `Sem padrão` | Reflectância TOA, [0,1] | Imagem já corrigida pelo ângulo solar. Não fornecer DN, radiância ou reflectância de superfície existente. |
| `parameters` | Obrigatório na reutilização de ajuste | `None` | Objeto de parâmetros | Usar resultado["parameters"] obtido com return_diagnostics=True no mesmo método. Omitir sun_elevation, percentile e assumed_dark_surface nessa rota; ajustar uma vez na cena e reutilizar nos blocos. |
| `mask` | Opcional; recomendado | `None` | Booleano | Máscara de exclusão de pixels, alinhada às imagens: True rejeita o pixel (NaN); False permite a avaliação, sujeita às validações do método. Não fornecer a banda QA bruta. A origem da máscara deste exemplo está descrita na seção de leitura. |
| `return_diagnostics` | Opcional | `False` | Booleano | False retorna somente a imagem; True retorna dicionário com: reflectance, parameters, method, units, invalid_mask. A máscara final pode incluir rejeições adicionais à máscara de entrada. |


In [ ]:
from pathlib import Path
import numpy as np
from pylstemp import optical_correction
from pylstemp.scene import LandsatScene

pasta = Path("CAMINHO_DA_CENA")
saida = Path("resultados/chavez_dos")
saida.mkdir(parents=True, exist_ok=True)
banda = 4  # banda optica
percentil = 0.01
reflectancia_alvo_escuro = 0.01
diagnosticos = True

## Ler e calibrar a imagem

`LandsatScene` confere os arquivos/grade da cena e le o MTL. `calibrated()` retorna bandas opticas, BT10/BT11 em K e mascara QA booleana. As entradas deste exemplo sao reflectancias TOA.

Neste leitor, B4/B5 estao disponiveis; B2 aparece quando presente na cena. Para outra banda, forneca a imagem TOA correspondente diretamente a funcao.

### Máscara usada neste exemplo

`mask` é uma máscara **de exclusão**, não uma máscara de seleção: `True` rejeita o pixel e `False` permite o cálculo. O leitor usa QA_PIXEL para fill, nuvem dilatada, cirrus de alta confiança, nuvem, sombra e neve; QA_RADSAT para saturação óptica B2/B4/B5 e oclusão de terreno; e verifica SZA quando disponível. A opção `exclude_water=True` também exclui água (bit7 de QA_PIXEL). Valores não finitos ou fora do domínio podem ser rejeitados adicionalmente pelo cálculo. `mask=None` em uma chamada direta não cria automaticamente uma máscara de nuvens.


In [ ]:
cena = LandsatScene(pasta)
opticas, bt, mascara = cena.calibrated(optical_quantity="reflectance")

## Aplicar o metodo

Ajustar uma vez na cena inteira e reutilizar parametros em blocos. Um alvo escuro incompatível gera erro; nao forcar ajuste por clipping. Fonte TM, transferencia OLI experimental; nao equivale a LaSRC nem corrige a atmosfera termica.

In [ ]:
resultado = optical_correction(
    # Método
    method='chavez-dos',

    # Reflectância TOA
    toa_reflectance=opticas[banda],

    # Ajuste na cena completa
    sun_elevation=float(cena.metadata['SUN_ELEVATION']),
    percentile=percentil,
    assumed_dark_surface=reflectancia_alvo_escuro,

    # Exclusão de pixels
    mask=mascara,

    # Saída
    return_diagnostics=diagnosticos,
)
imagem = resultado['reflectance'] if diagnosticos else resultado


## Salvar a imagem

A saida preserva a grade original e e gravada como float32 com NoData NaN. A unidade da emissividade/reflectancia e adimensional; PWV usa g/cm2; BT usa K e LST respeita `unit`.

In [ ]:
cena.write_geotiff(saida / "chavez_dos.tif", imagem, units='dimensionless')
cena.close()
print("Pixels validos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saida:", saida)

## Referencia e limites

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).